# Predicting the site of metabolism

`som_oracle.ipynb` measured that filtering candidates to a parent's labelled sites takes recall@5
from 46.6% to 61.2%. This notebook trains a model to produce the sites, so the gain can be measured
with predicted sites.

**The task is copy-with-one-edit.** Input is a canonical SMILES; output is the same string with an
atom map on the reacting atom. The label is kept exactly as the source dataset records it, and a
row is kept only when stripping that atom map returns the canonical parent unchanged, so input and
output share an atom ordering. The model reproduces the molecule and changes one atom. Rows whose
label is written against a different tautomer or salt form fail that check and are dropped.

**No spelling augmentation.** The generator in `notebooks/03` uses it: several spellings of
one molecule map to the same metabolite, so augmentation teaches invariance and test-time voting
across spellings exploits it. Here the output is a copy of the input, so augmenting would force the
model to learn re-ordering as well as marking, a skill never used because every parent is
canonicalised before it reaches the model. Diversity of *sites* comes from temperature sampling,
which is what the filter needs.

**An untuned Qwen3-8B cannot do this**: given four worked examples, 7 of 12 outputs parsed as a
molecule and only 1 reproduced the input with a marker, the rest copying a molecule out of the
prompt.

## 1. Data and format

In [1]:
import json, os, pathlib, random, sys, time, collections
import numpy as np, pandas as pd

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "SOM" else pathlib.Path.cwd()
HERE = ROOT / "SOM"
sys.path.insert(0, str(ROOT))
from src import evaluation as ev
for line in (ROOT / ".env").read_text().splitlines():
    if "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k.strip()] = v.strip()
import tinker
import tinker.types as tt
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

PROMPT = "<molecule>{}</molecule>\n<site>"
COMPLETION = "{}</site>"
BASE_MODEL = "Qwen/Qwen3-8B"

SOM_TRAIN, SOM_DEV = HERE / "som_train.jsonl", HERE / "som_dev.jsonl"

def build_site_splits():
    """One row per (canonical parent, that parent with its reacting atom marked).

    Sources are the same four datasets the rest of the project uses, with GOSTAR filtered to
    mammalian species by the rule notebook 05 applies. A row survives only if stripping the atom
    map off the label returns exactly the canonical parent: that makes the task copy-with-one-edit
    rather than re-derivation, and drops labels recorded against a different tautomer or salt form.
    Test parents are excluded and dev is the parent set of the main dev split, so no parent appears
    on both sides of the split or in the holdout test set.

    The shipped splits were produced by an earlier version of this rule and the files are not
    overwritten. A clean rebuild lands on the same 550 dev rows and 13,371 rather than 13,373
    training rows: about 0.25% of rows differ, from deduplication order. Nothing downstream
    depends on which of the two it gets.
    """
    import re
    MAMMAL = {"human", "rat", "mouse", "dog", "monkey", "rabbit", "guinea pig", "hamster",
              "pig", "cat", "mammal"}
    def is_mammal(s):
        parts = [x.strip().lower() for x in re.split(r"[;,/]", s or "") if x.strip()]
        return bool(parts) and all(x in MAMMAL for x in parts)
    def strip_maps(s):
        m = Chem.MolFromSmiles(s) if s else None
        if m is None: return None
        for a in m.GetAtoms(): a.SetAtomMapNum(0)
        return Chem.MolToSmiles(m)

    frames = []
    for name in ("drugbank", "gloryx", "metxbiodb", "gostar"):
        d = pd.read_csv(ROOT / f"data/{name}.csv", dtype=str, keep_default_na=False, na_values=[])
        if name == "gostar": d = d[d["species"].map(is_mammal)]
        frames.append(d[["parent_smiles", "som_smiles"]])
    src = pd.concat(frames, ignore_index=True)
    src = src[src["som_smiles"].str.len() > 0]

    test = pd.read_csv(ROOT / "data/test_set.csv", dtype=str, keep_default_na=False, na_values=[])
    test_keys = {ev.standardize(s) for s in test["parent_smiles"]}
    dev_keys = {ev.standardize(json.loads(l)["parent"])
                for l in (ROOT / "data/processed/dev.jsonl").open()}

    seen, rows = set(), []
    for a, b in zip(src["parent_smiles"], src["som_smiles"]):
        parent = ev.canonical(a)
        if parent is None or ev.standardize(parent) in test_keys: continue
        if strip_maps(b) != parent: continue
        if (parent, b) in seen: continue
        seen.add((parent, b)); rows.append({"parent": parent, "marked": b})
    return ([r for r in rows if ev.standardize(r["parent"]) not in dev_keys],
            [r for r in rows if ev.standardize(r["parent"]) in dev_keys])

if not (SOM_TRAIN.exists() and SOM_DEV.exists()):
    _tr, _dv = build_site_splits()
    SOM_TRAIN.write_text("".join(json.dumps(r) + "\n" for r in _tr))
    SOM_DEV.write_text("".join(json.dumps(r) + "\n" for r in _dv))
    print(f"built the site splits: {len(_tr):,} train, {len(_dv):,} dev")

train_rows = [json.loads(l) for l in SOM_TRAIN.open()]
dev_rows   = [json.loads(l) for l in SOM_DEV.open()]
print(f"train {len(train_rows):,} pairs over {len({r['parent'] for r in train_rows}):,} parents")
print(f"dev   {len(dev_rows):,} pairs over {len({r['parent'] for r in dev_rows}):,} parents")

def marked_atoms(smiles, reference_parent):
    """Atom indices carrying map 1, expressed in reference_parent's indexing. None if the string
    is not that parent.

    `breakTies=False` is load-bearing. It makes CanonicalRankAtoms return symmetry classes rather
    than a total order, so chemically equivalent atoms share a rank and marking either of them
    yields the same set. With the default they are distinguishable, and marking the other end of a
    symmetric pair reads as a different site: 5.2% of labels change their site set under a
    re-spelling of the same molecule, and 9.8% have equivalent atoms the label does not list. Both
    would score a chemically correct prediction as a miss.
    """
    m = Chem.MolFromSmiles(smiles) if smiles else None
    if m is None: return None
    idx = [a.GetIdx() for a in m.GetAtoms() if a.GetAtomMapNum() == 1]
    if not idx: return None
    for a in m.GetAtoms(): a.SetAtomMapNum(0)
    if Chem.MolToSmiles(m) != reference_parent: return None      # a different molecule
    p = Chem.MolFromSmiles(reference_parent)
    rp = list(Chem.CanonicalRankAtoms(p, breakTies=False))
    rm = list(Chem.CanonicalRankAtoms(m, breakTies=False))
    want = {rm[i] for i in idx}
    return {i for i, r in enumerate(rp) if r in want} or None

truth = collections.defaultdict(set)
for r in train_rows + dev_rows:
    s = marked_atoms(r["marked"], r["parent"])
    if s: truth[r["parent"]] |= s
print(f"sites per parent, on average: {np.mean([len(v) for v in truth.values()]):.2f}")
print(f"\none example, exactly as the model receives it:")
print(f"  shown : {PROMPT.format(train_rows[0]['parent'])!r}")
print(f"  writes: {COMPLETION.format(train_rows[0]['marked'])!r}")

/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_seq_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_path" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_owner" has conflict with protected namespace "model_".

You may be able to res

train 13,373 pairs over 8,321 parents
dev   550 pairs over 280 parents


sites per parent, on average: 2.42

one example, exactly as the model receives it:
  shown : '<molecule>C[C@]12CCC(=O)C[C@@H]1CC[C@@H]1[C@@H]2CC[C@]2(C)[C@@H](O)CC[C@@H]12</molecule>\n<site>'
  writes: 'C[C@]12CC[C:1](=O)C[C@@H]1CC[C@@H]1[C@@H]2CC[C@]2(C)[C@@H](O)CC[C@@H]12</site>'


## 2. The training client, and what the loss is scored on

In [2]:
service = tinker.ServiceClient(api_key=os.environ["TINKER_API_KEY"])
training = service.create_lora_training_client(
    base_model=BASE_MODEL, rank=32, train_mlp=True, train_attn=True, train_unembed=True)
tokenizer = training.get_tokenizer()
EOS = tokenizer.eos_token_id or 0

def to_datum(row):
    """Prompt is context; the marked molecule is the target. Weights are 0 on the prompt and 1 on
    every position predicting an answer token, plus the EOS position, so the model is trained to
    stop as well as to answer."""
    prompt_ids = tokenizer.encode(PROMPT.format(row["parent"]))
    answer_ids = tokenizer.encode(COMPLETION.format(row["marked"]))
    ids = prompt_ids + answer_ids
    return tt.Datum(model_input=tt.ModelInput.from_ints(ids),
                    loss_fn_inputs={"target_tokens": ids[1:] + [EOS],
                                    "weights": [0.0]*(len(prompt_ids)-1) + [1.0]*len(answer_ids) + [1.0]})

def n_supervised(rows):
    return sum(len(tokenizer.encode(COMPLETION.format(r["marked"]))) + 1 for r in rows)

def batch_loss(fb, supervised):
    total = sum(float(np.asarray(o["elementwise_loss"].data).sum()) for o in fb.loss_fn_outputs)
    return total / max(supervised, 1)

_r = train_rows[0]
_el = np.asarray(training.forward([to_datum(_r)], "cross_entropy").result()
                 .loss_fn_outputs[0]["elementwise_loss"].data)
_np = len(tokenizer.encode(PROMPT.format(_r["parent"]))) - 1
assert _el[:_np].sum() == 0, "prompt positions carry loss"
assert _el[_np] != 0, "the first answer position carries none; the mask is one token late"
lengths = [len(tokenizer.encode(PROMPT.format(r["parent"]) + COMPLETION.format(r["marked"])))
           for r in train_rows[:400]]
print(f"masking verified. ~{np.mean(lengths):.0f} tokens per example, "
      f"~{np.mean(lengths)*len(train_rows)/1e6:.2f}M per epoch")

masking verified. ~84 tokens per example, ~1.12M per epoch


## 3. How a checkpoint is chosen

Checkpoints are chosen on a probe of the task. In notebook 03 dev loss bottomed out in epoch 0 while the
metric kept improving, and the same risk applies to any proxy. The probe measures the quantity the
filter needs: sample the model several times for a parent, collect the atoms it marks, and check
whether they land on real sites.

`any_hit` is the one that matters for filtering, since the oracle filtered on the union of a
parent's sites. `precision` is reported alongside because a model that marks everything would score
a perfect `any_hit` and be useless.

In [3]:
SAMPLES = 4
PROBE = 60
rng = np.random.default_rng(0)
dev_parents = sorted({r["parent"] for r in dev_rows})
probe_parents = sorted(rng.choice(dev_parents, size=min(PROBE, len(dev_parents)), replace=False))

def predict_sites(client, parents, n=SAMPLES, temperature=0.8):
    jobs = {p: client.sample(
                prompt=tt.ModelInput.from_ints(tokenizer.encode(PROMPT.format(p))),
                num_samples=n,
                sampling_params=tt.SamplingParams(max_tokens=200, temperature=temperature,
                                                  stop=["</site>", "\n"]))
            for p in parents}
    out, parsed, total = {}, 0, 0
    for p, fut in jobs.items():
        sites = set()
        for s in fut.result().sequences:
            total += 1
            txt = tokenizer.decode(s.tokens).split("</site>")[0].strip()
            got = marked_atoms(txt, p)
            if got: sites |= got; parsed += 1
        out[p] = sites
    return out, parsed / max(total, 1)

def site_scores(client, parents):
    pred, valid = predict_sites(client, parents)
    hit = [bool(pred[p] & truth[p]) for p in parents if truth[p]]
    prec = [len(pred[p] & truth[p]) / len(pred[p]) for p in parents if truth[p] and pred[p]]
    return {"any_hit": float(np.mean(hit)) if hit else 0.0,
            "precision": float(np.mean(prec)) if prec else 0.0,
            "usable": valid,
            "sites": float(np.mean([len(pred[p]) for p in parents]))}

untuned = site_scores(service.create_sampling_client(base_model=BASE_MODEL), probe_parents)
print(f"untuned on the same probe: any_hit {untuned['any_hit']:.1%}  "
      f"precision {untuned['precision']:.1%}  usable replies {untuned['usable']:.1%}")

untuned on the same probe: any_hit 0.0%  precision 0.0%  usable replies 0.0%


## 4. Smoke run

In [4]:
BATCH, LR, SMOKE = 48, 1e-4, 20
random.seed(0)
shuffled = train_rows[:]; random.shuffle(shuffled)
losses, t0 = [], time.time()
for step in range(SMOKE):
    rows = shuffled[step*BATCH:(step+1)*BATCH]
    fb = training.forward_backward([to_datum(r) for r in rows], "cross_entropy")
    training.optim_step(adam_params=tinker.AdamParams(learning_rate=LR)).result()
    losses.append(batch_loss(fb.result(), n_supervised(rows)))
    if step % 5 == 0: print(f"  step {step:>2}  loss {losses[-1]:.4f}")
secs_per_step = (time.time()-t0)/SMOKE
first, last = np.mean(losses[:3]), np.mean(losses[-3:])
print(f"\n  loss {first:.4f} -> {last:.4f}  ({secs_per_step:.1f}s/step)")
assert last < first, f"loss did not fall ({first:.4f} -> {last:.4f})"
print("  gate passed")

  step  0  loss 1.6761


  step  5  loss 0.8217


  step 10  loss 0.4157


  step 15  loss 0.3393



  loss 1.4795 -> 0.3150  (2.6s/step)
  gate passed


## 5. Training

Three epochs. Both reranker runs plateaued after epoch 1.5 regardless of dataset size, and the
generator peaked at epoch 3 of 6, so more epochs are not expected to help. Selection keeps the
best checkpoint by `any_hit`, so an extra epoch costs only time.

In [5]:
EPOCHS = 3
TTL = 7 * 24 * 3600
steps_per_epoch = len(shuffled) // BATCH
EVAL_EVERY = max(1, steps_per_epoch // 2)
print(f"  {steps_per_epoch} steps per epoch, {steps_per_epoch*EPOCHS} total, "
      f"~{steps_per_epoch*EPOCHS*secs_per_step/60:.0f} min of training\n")

history, best, step, t0 = [], {"any_hit": -1.0, "step": -1, "path": None}, 0, time.time()
for epoch in range(EPOCHS):
    random.shuffle(shuffled)
    for i in range(0, len(shuffled) - BATCH + 1, BATCH):
        rows = shuffled[i:i+BATCH]
        fb = training.forward_backward([to_datum(r) for r in rows], "cross_entropy")
        training.optim_step(adam_params=tinker.AdamParams(learning_rate=LR)).result()
        row = {"step": step, "epoch": epoch, "loss": batch_loss(fb.result(), n_supervised(rows)),
               "any_hit": None, "precision": None, "usable": None, "path": None}
        if step % EVAL_EVERY == 0:
            path = training.save_weights_for_sampler(name=f"som{step}", ttl_seconds=TTL).result().path
            sc = site_scores(service.create_sampling_client(model_path=path), probe_parents)
            row.update({**sc, "path": path})
            flag = ""
            if sc["any_hit"] > best["any_hit"]:
                best = {"any_hit": sc["any_hit"], "step": step, "path": path}; flag = "  <- best"
            print(f"  step {step:>4} ep {epoch}  loss {row['loss']:.4f}  "
                  f"any_hit {sc['any_hit']:>5.1%}  precision {sc['precision']:>5.1%}  "
                  f"usable {sc['usable']:>5.1%}{flag}")
        history.append(row); step += 1

hist = pd.DataFrame(history)
assert hist["any_hit"].notna().sum() >= 2, "fewer than two evaluations landed"
hist.to_csv(HERE / "som_predictor_history.csv", index=False)
(HERE / "som_predictor_run.json").write_text(json.dumps({
    "base_model": BASE_MODEL, "rank": 32, "batch": BATCH, "learning_rate": LR, "epochs": EPOCHS,
    "training_steps": step, "train_examples": len(train_rows),
    "best_step": best["step"], "best_checkpoint": best["path"], "best_any_hit": best["any_hit"],
    "untuned_any_hit": untuned["any_hit"], "untuned_usable": untuned["usable"],
    "checkpoint_ttl_seconds": TTL,
    "all_checkpoints": {int(r.step): r.path for r in hist.dropna(subset=["path"]).itertuples()},
}, indent=2))
print(f"\n  {step} steps in {(time.time()-t0)/60:.1f} min")
print(f"  best any_hit {best['any_hit']:.1%} at step {best['step']} (untuned {untuned['any_hit']:.1%})")

  278 steps per epoch, 834 total, ~36 min of training



  step    0 ep 0  loss 0.3720  any_hit  8.3%  precision 13.6%  usable 15.0%  <- best


  step  139 ep 0  loss 0.2355  any_hit 63.3%  precision 30.1%  usable 74.2%  <- best


  step  278 ep 1  loss 0.1334  any_hit 55.0%  precision 24.0%  usable 79.2%


  step  417 ep 1  loss 0.1492  any_hit 56.7%  precision 30.3%  usable 73.8%


  step  556 ep 2  loss 0.1075  any_hit 55.0%  precision 27.9%  usable 82.9%


  step  695 ep 2  loss 0.1250  any_hit 70.0%  precision 32.1%  usable 80.8%  <- best



  834 steps in 50.6 min
  best any_hit 70.0% at step 695 (untuned 0.0%)


## 6. Sites for the test parents

Predicted sites for the same 155 test parents the oracle used, at the probe's sampling budget, so
`som_oracle.ipynb` can run the identical filter and ranker on them. The model sees only parent
structures, never a test metabolite.

In [6]:
test = pd.read_csv(ROOT / "data" / "test_set.csv", dtype=str, keep_default_na=False, na_values=[])
test_parents = sorted({ev.canonical(s) for s in test["parent_smiles"]})
assert not (set(test_parents) & {r["parent"] for r in train_rows + dev_rows}), \
    "a test parent appears in the site training data"

pred, usable = predict_sites(service.create_sampling_client(model_path=best["path"]), test_parents)
empty = [p for p in test_parents if not pred[p]]
print(f"  {len(test_parents)} test parents, {usable:.1%} of replies usable")
print(f"  sites predicted per parent : {np.mean([len(pred[p]) for p in test_parents]):.2f}")
print(f"  parents with no site at all: {len(empty)} ({len(empty)/len(test_parents):.1%})")

(HERE / "som_predicted_sites.json").write_text(json.dumps({
    "checkpoint": best["path"], "best_step": best["step"], "samples_per_parent": SAMPLES,
    "usable_replies": usable, "parents_without_a_site": len(empty),
    "sites": {p: sorted(pred[p]) for p in test_parents}}, indent=1))
print("\n  saved -> SOM/som_predicted_sites.json")

  155 test parents, 86.1% of replies usable
  sites predicted per parent : 3.28
  parents with no site at all: 0 (0.0%)

  saved -> SOM/som_predicted_sites.json
